In [2]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data_tag = "ablation-v3_0510"

tasks = [
    "bace",
    "smol-property_prediction-bbbp",
    "smol-property_prediction-clintox",
    "smol-property_prediction-hiv",
    "smol-property_prediction-sider",
    "smol-property_prediction-esol",
    "smol-property_prediction-lipo",
    "qm9_homo",
    "forward_reaction_prediction",
    "retrosynthesis",
    "reagent_prediction",
]

path_template = "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{}_{}_0219"

In [ ]:
trainset_name = f'/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_{data_tag}'
train_datasets = {}
for task in tasks:
    path = path_template.format("train", task)
    train_datasets[task] = datasets.load_from_disk(path)

# concat train_datasets
list_train_datasets = []
for task in tasks:
    list_train_datasets.append(train_datasets[task])

concat_train_datasets = datasets.concatenate_datasets(list_train_datasets)
concat_train_datasets.save_to_disk(trainset_name)

# load trainset
trainset = datasets.load_from_disk(trainset_name)


Saving the dataset (11/11 shards): 100%|██████████| 561369/561369 [01:12<00:00, 7736.39 examples/s] 


In [19]:
trainset = datasets.load_from_disk(trainset_name)
trainset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 561369
})

In [20]:
trainset = datasets.load_from_disk(trainset_name)
list(set(trainset['task']))

['bace',
 'qm9_homo',
 'smol-property_prediction-sider',
 'smol-property_prediction-bbbp',
 'smol-property_prediction-hiv',
 'smol-property_prediction-lipo',
 'smol-property_prediction-esol',
 'reagent_prediction',
 'forward_reaction_prediction',
 'retrosynthesis',
 'smol-property_prediction-clintox']

In [11]:
alchemy_homo_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy_homo_0405'
aqsol_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_aqsol_0405'
orderly_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly_forward_0509'
orderly_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly_retro_0509'
presto_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-forward_reaction_prediction_0405'
presto_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-retrosynthesis_0405'

# load data
alchemy_data = datasets.load_from_disk(alchemy_homo_path)
aqsol_data = datasets.load_from_disk(aqsol_path)
orderly_forward_data = datasets.load_from_disk(orderly_forward_path)
presto_forward_data = datasets.load_from_disk(presto_forward_path)
orderly_retro_data = datasets.load_from_disk(orderly_retro_path)
presto_retro_data = datasets.load_from_disk(presto_retro_path)

ood_test_data = datasets.concatenate_datasets(
    [
        alchemy_data,
        aqsol_data,
        orderly_forward_data,
        orderly_retro_data,
        presto_forward_data,
        presto_retro_data,
    ]
)

In [12]:
testset_name = f'/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_{data_tag}'
test_datasets = {}
for task in tasks:
    path = path_template.format("test", task)
    test_datasets[task] = datasets.load_from_disk(path)

# concat test_datasets
list_test_datasets = []
for task in tasks:
    list_test_datasets.append(test_datasets[task])


list_test_datasets.append(ood_test_data)

concat_test_datasets = datasets.concatenate_datasets(list_test_datasets)
concat_test_datasets.save_to_disk(testset_name)

Saving the dataset (1/1 shards): 100%|██████████| 17605/17605 [00:01<00:00, 9055.52 examples/s] 


In [17]:
testset = datasets.load_from_disk(testset_name)
testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 17605
})

In [18]:
list(set(testset['task']))

['bace',
 'orderly-forward_reaction_prediction',
 'qm9_homo',
 'smol-property_prediction-sider',
 'aqsol-logS',
 'smol-property_prediction-bbbp',
 'smol-property_prediction-hiv',
 'smol-property_prediction-lipo',
 'alchemy_homo',
 'smol-property_prediction-esol',
 'reagent_prediction',
 'forward_reaction_prediction',
 'presto-forward_reaction_prediction',
 'retrosynthesis',
 'smol-property_prediction-clintox',
 'presto-retrosynthesis',
 'orderly-retrosynthesis']